In [1]:
import pandas as pd

In [2]:
data = pd.read_csv('../../data/train.csv') 
data = data.dropna(subset=['Context','Response'])
data_unique = data.drop_duplicates(subset=['Context','Response']) 
test_data = data_unique.sample(n=10,random_state=42) 
train_data = data_unique.drop(test_data.index) 
train_data.to_csv('../../data/train_unique_e2p2.csv',index=False) 
test_data.to_csv('../../data/test_unique_e2p2.csv',index=False)

In [3]:
from nltk.corpus import stopwords
import string
stop_words = set(stopwords.words('english'))

In [4]:
#Nettoyage des données
def text_process(mess):
    lower_mess = mess.lower()
    nopunc = [char for char in lower_mess if char not in string.punctuation]
    nopunc = ''.join(nopunc)
    clean_mess = [word for word in nopunc.split() if word not in stop_words]
    return clean_mess

In [5]:
from sklearn.metrics.pairwise import cosine_similarity
train_data = pd.read_csv('../../data/train_unique_e2p2.csv')
test_data = pd.read_csv('../../data/test_unique_e2p2.csv')
responses_train = train_data["Response"].astype(str).str.strip().drop_duplicates().reset_index(drop=True)

In [12]:
#Méthode 2 : Utilisation de TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(analyzer=text_process)
X_train = vectorizer.fit_transform(responses_train)
def k_response(question,k):
    X_question = vectorizer.transform([question])
    cosine_similarities = cosine_similarity(X_question,X_train).flatten()
    top_k_indices = cosine_similarities.argsort()[-k:][::-1]
    #return responses_train.iloc[top_k_indices].tolist()
    results = []
    for idx in top_k_indices:
        results.append((responses_train.iloc[idx],cosine_similarities[idx]))
    return results

In [13]:
#Test de la méthode
questions = test_data['Context'].tolist()
responses = test_data['Response'].tolist()
for question,response in zip(questions,responses):
    print(f"Question: {question}")
    print(f"Actual Response: {response}")
    print("Top 3 responses:")
    list = k_response(question,3)
    for response,score in list:
        print(f"- (Cosine Similarity: {score:.4f}) {response}")
    print("\n")

Question: What makes a healthy marriage last?
Actual Response: This answer varies based on you relationship. However, I do believe their are some basic fundamental areas that are beneficial for a healthy marriage:1.) Effective Communication2.) Trust3.) Love/Passion4.) Loyalty. 5.) Unconditional Positive Regard. Everyone has their favorite qualities they feel best fit a marriage. However, these are what I think are great starting points. 
Top 3 responses:
- (Cosine Similarity: 0.3016) I get it. Your husband tells you that he's not in love with you, but oops, he's changed his mind and will tolerate you for a while longer? Excuse me? My Dear, it's okay if you expect more than that from a marriage. Maybe the question has shifted from whether he is happy in the marriage to whether you are happy in the marriage. You say you love this man,  who makes you "feel like nothing". I say it might be time to sit down with an individual therapist and look objectively at your marriage and whether it's 

In [10]:
# Méthode 2 : Utilisation de BERT
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
response_bert = model.encode(responses_train.tolist())
def k_response_bert(question,k):
    question_embedding = model.encode([question])
    cosine_similarities = cosine_similarity(question_embedding,response_bert).flatten()
    top_k_indices = cosine_similarities.argsort()[-k:][::-1]
    #return responses_train.iloc[top_k_indices].tolist()
    results = []
    for idx in top_k_indices:
        results.append((responses_train.iloc[idx],cosine_similarities[idx]))
    return results

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
# Test de la méthode BERT
questions = test_data['Context'].tolist()
responses = test_data['Response'].tolist()
for question,response in zip(questions,responses):
    print(f"Question: {question}")
    print(f"Actual Response: {response}")
    print("Top 3 responses (BERT):")
    list = k_response_bert(question,3)
    for response,score in list:
        print(f"- (Cosine Similarity: {score:.4f}) {response}")
    print("\n")

Question: What makes a healthy marriage last?
Actual Response: This answer varies based on you relationship. However, I do believe their are some basic fundamental areas that are beneficial for a healthy marriage:1.) Effective Communication2.) Trust3.) Love/Passion4.) Loyalty. 5.) Unconditional Positive Regard. Everyone has their favorite qualities they feel best fit a marriage. However, these are what I think are great starting points. 
Top 3 responses (BERT):
- (Cosine Similarity: 0.7367) This answer varies based on you relationship. However, I do believe their are some basic fundamental areas that are beneficial for a healthy marriage:1.) Effective Communication2.) Trust3.) Love/Passion4.) Loyalty. 5.) Unconditional Positive Regard. Everyone has their favorite qualities they feel best fit a marriage. However, these are what I think are great starting points.
- (Cosine Similarity: 0.6675) Thank you for your question.  A good Marriage is one that takes hard work and commitment.  Being